
# Notebook 04 — The Knowledge Store (curation that earns trust)

This is the heart of the workshop. Teams that see low Genie adoption almost always **skipped this step**: they attach tables, stuff every rule into one giant instructions blob, and hope. The **Knowledge Store** is the structured alternative — and it's what makes an agent trustworthy.

**The anti-pattern (what not to do):** one enormous `text_instructions` blob holding every formula, join, and classification. It eats the context window, invites contradictions, and Genie applies it inconsistently.

**The fix — structured components**, each doing one job:

| Component | What it pins | Manufacturing example |
|---|---|---|
| **Column configs** | descriptions, **synonyms**, hidden columns | `oee_score` → "OEE"; hide `unit_serial_vin` (VIN) |
| **Joins** | the correct join paths | `production_events` → `production_lines` (many-to-one) |
| **Measures** | KPI formulas | Scrap Rate = `SUM(scrap_count)/SUM(units_produced)` |
| **Filters** | reusable conditions | Active lines only (`status = 'Active'`) |
| **Fields / expressions** | row-level derivations | Defect category from `defect_code` |
| **Example SQL** | question → SQL patterns | "defect rate above 5%" with the right HAVING |
| **Text instructions** | *only* clarifications & formatting | ask for a time range; format % to 2 dp |

**Priority order:** measures/filters/fields → example SQL → text **last**. Genie's own docs show example SQL is the single biggest accuracy jump.

This notebook builds all of that **programmatically** via `serialized_space` (v2) and creates the **primary agent** (`genie_space_id`) that later notebooks benchmark and deploy.

> **Schema note.** The `serialized_space` shape here matches the Genie Workbench source. Before your session, confirm it against a live agent: `GET /api/2.0/genie/spaces/<id>?include_serialized_space=true`. If a field differs, adjust the builders below — the structure is what matters.

**Before you start:** run notebooks **02** (data) and **03** (baseline).

**Compute:** Serverless.

In [ ]:
%run ./00_workshop_config

In [ ]:
from databricks.sdk import WorkspaceClient
import re
import json
import uuid
import requests

w = WorkspaceClient()
host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}


def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"


def new_id():
    # Genie requires 32-char lowercase hex ids (no hyphens), unique within the space.
    return uuid.uuid4().hex


warehouse_id = None
for wh in w.warehouses.list():
    if str(wh.state).upper() in ("RUNNING", "STARTING"):
        warehouse_id = wh.id
        break
if not warehouse_id:
    whs = list(w.warehouses.list())
    warehouse_id = whs[0].id if whs else None
if not warehouse_id:
    raise RuntimeError("No SQL warehouse found. Create or start one, then re-run.")
print("Warehouse:", warehouse_id)

## 1. Column configs — descriptions, synonyms, and hidden columns

Teach Genie the business vocabulary (**synonyms**) and hide sensitive columns. Here we hide `unit_serial_vin` (a VIN — sensitive) and give cryptic columns friendly names. All matching flags are off for a hidden column; entity-matching requires format-assistance, so we set both for the text columns users filter on.

In [ ]:
def col(name, description=None, synonyms=None, exclude=False,
        entity_matching=False, format_assist=False):
    return {
        "column_name": name,
        "description": [description] if description else [],
        "synonyms": synonyms or [],
        "exclude": exclude,
        "enable_entity_matching": entity_matching,
        "enable_format_assistance": format_assist or entity_matching,
        "get_example_values": False,
        "build_value_dictionary": False,
    }


column_configs_by_table = {
    "production_events": [
        col("unit_serial_vin", description="17-char VIN, sensitive — hidden from the agent.", exclude=True),
        col("event_type", synonyms=["event"],
            description="One of: unit_produced, defect_detected, inspection_passed, downtime_start, scrap, rework_completed."),
        col("defect_code", synonyms=["defect type"], description="Set only when event_type = 'defect_detected'."),
    ],
    "quality_metrics_daily": [
        col("oee_score", synonyms=["OEE", "overall equipment effectiveness"],
            description="Overall Equipment Effectiveness, 0-1 scale; multiply by 100 for percent."),
        col("first_pass_yield", synonyms=["FPY", "first pass yield"],
            description="0-1 scale; multiply by 100 for percent."),
        col("scrap_count", synonyms=["scrap"]),
        col("downtime_minutes", synonyms=["downtime"]),
    ],
    "plants": [
        col("state", synonyms=["plant state"], entity_matching=True),
        col("plant_name", synonyms=["plant", "facility"], entity_matching=True),
    ],
    "production_lines": [
        col("status", synonyms=["line status"],
            description="STATIC attribute: 'Active' or 'Maintenance'. No date — never filter it by date."),
        col("product_type", synonyms=["product"]),
    ],
}

ALL_TABLES = ["plants", "production_lines", "operators", "production_events",
              "quality_metrics_daily", "safety_incidents", "equipment_feedback"]


def table_entry(short):
    cfgs = sorted(column_configs_by_table.get(short, []), key=lambda c: c["column_name"])
    e = {"identifier": f"{fqn}.{short}"}
    if cfgs:
        e["column_configs"] = cfgs
    return e


tables = sorted([table_entry(t) for t in ALL_TABLES], key=lambda e: e["identifier"])
print(f"{len(tables)} tables; hidden: unit_serial_vin; synonyms on OEE, FPY, plant, state, ...")

## 2. Joins — pin the correct paths

Each join records the relationship type in the required `--rt=FROM_RELATIONSHIP_TYPE_*--` marker. Now Genie never has to guess how the fact tables connect to plants, lines, and operators.

In [ ]:
def join_spec(l_id, l_alias, l_col, r_id, r_alias, r_col, rel):
    return {
        "id": new_id(),
        "left": {"identifier": l_id, "alias": l_alias},
        "right": {"identifier": r_id, "alias": r_alias},
        "sql": [f"`{l_alias}`.`{l_col}` = `{r_alias}`.`{r_col}`",
                f"--rt=FROM_RELATIONSHIP_TYPE_{rel}--"],
        "comment": [],
        "instruction": [],
    }


T = lambda s: f"{fqn}.{s}"
joins = [
    join_spec(T("production_events"), "pe", "production_line_id", T("production_lines"), "pl", "line_id", "MANY_TO_ONE"),
    join_spec(T("production_events"), "pe", "operator_id", T("operators"), "op", "operator_id", "MANY_TO_ONE"),
    join_spec(T("production_lines"), "pl", "plant_id", T("plants"), "pln", "plant_id", "MANY_TO_ONE"),
    join_spec(T("quality_metrics_daily"), "qm", "production_line_id", T("production_lines"), "pl", "line_id", "MANY_TO_ONE"),
    join_spec(T("quality_metrics_daily"), "qm", "plant_id", T("plants"), "pln", "plant_id", "MANY_TO_ONE"),
    join_spec(T("safety_incidents"), "si", "production_line_id", T("production_lines"), "pl", "line_id", "MANY_TO_ONE"),
    join_spec(T("equipment_feedback"), "ef", "production_line_id", T("production_lines"), "pl", "line_id", "MANY_TO_ONE"),
]
print(f"{len(joins)} joins defined")

## 3. Measures, filters, and fields (SQL snippets)

**Measures** are the KPI formulas — the math Genie should never re-invent. **Filters** are reusable conditions. **Fields/expressions** are row-level derivations. This is where the logic that used to live in the giant text blob actually belongs.

In [ ]:
def snippet(display_name, sql, synonyms=None, instruction=None):
    return {
        "id": new_id(),
        "display_name": display_name,
        "sql": [sql],
        "synonyms": synonyms or [],
        "instruction": [instruction] if instruction else [],
        "comment": [],
    }


measures = [
    snippet("Average OEE %", "ROUND(AVG(quality_metrics_daily.oee_score) * 100, 2)",
            synonyms=["OEE", "overall equipment effectiveness"],
            instruction="Average Overall Equipment Effectiveness as a percentage."),
    snippet("Average First Pass Yield %", "ROUND(AVG(quality_metrics_daily.first_pass_yield) * 100, 2)",
            synonyms=["FPY", "first pass yield"]),
    snippet("Scrap Rate %",
            "ROUND(100.0 * SUM(quality_metrics_daily.scrap_count) / NULLIF(SUM(quality_metrics_daily.units_produced), 0), 2)",
            synonyms=["scrap rate"],
            instruction="Scrap rate over a date range: use quality_metrics_daily, not production_events."),
    snippet("Defect Rate %",
            "ROUND(100.0 * SUM(CASE WHEN production_events.event_type = 'defect_detected' THEN 1 ELSE 0 END) "
            "/ NULLIF(SUM(CASE WHEN production_events.event_type = 'unit_produced' THEN 1 ELSE 0 END), 0), 2)",
            synonyms=["defect rate"]),
    snippet("Rework Rate %",
            "ROUND(100.0 * SUM(CASE WHEN production_events.event_type = 'rework_completed' THEN 1 ELSE 0 END) "
            "/ NULLIF(SUM(CASE WHEN production_events.event_type = 'defect_detected' THEN 1 ELSE 0 END), 0), 2)",
            synonyms=["rework rate"]),
]

filters = [
    snippet("Active lines only", "production_lines.status = 'Active'", synonyms=["active lines"],
            instruction="status is static (Active/Maintenance); never add a date filter to it."),
    snippet("Defects only", "production_events.event_type = 'defect_detected'", synonyms=["defects"]),
]

expressions = [
    snippet("Operator full name", "CONCAT(operators.first_name, ' ', operators.last_name)",
            synonyms=["operator name"]),
    snippet("Defect category",
            "CASE WHEN production_events.defect_code LIKE 'DEF-WELD-%' THEN 'Welding' "
            "WHEN production_events.defect_code LIKE 'DEF-PAINT-%' THEN 'Paint' "
            "WHEN production_events.defect_code LIKE 'DEF-FIT-%' THEN 'Fit & Finish' "
            "WHEN production_events.defect_code LIKE 'DEF-ELEC-%' THEN 'Electrical' "
            "WHEN production_events.defect_code LIKE 'DEF-STMP-%' THEN 'Stamping' ELSE 'Other' END",
            synonyms=["defect type category"]),
]
print(f"{len(measures)} measures, {len(filters)} filters, {len(expressions)} fields")

## 4. Example SQL — the highest-impact curation

A natural-language question paired with its correct SQL. Genie matches new questions to these patterns. Some carry **parameters** (`:year`) so one example covers many time ranges.

In [ ]:
def param(name, type_hint, description, default=None):
    p = {"name": name, "type_hint": type_hint, "description": [description]}
    if default is not None:
        p["default_value"] = {"values": [str(default)]}
    return p


def example(question, sql, parameters=None, usage_guidance=None):
    return {
        "id": new_id(),
        "question": [question],
        "sql": [sql],
        "parameters": sorted(parameters or [], key=lambda p: p["name"]),
        "usage_guidance": [usage_guidance] if usage_guidance else [],
    }


examples = [
    example(
        "What is the average OEE by plant for {year}?",
        f"SELECT p.plant_name, ROUND(AVG(q.oee_score) * 100, 2) AS avg_oee_pct "
        f"FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id "
        f"WHERE YEAR(CAST(q.date AS DATE)) = :year GROUP BY p.plant_name ORDER BY avg_oee_pct DESC",
        parameters=[param("year", "NUMBER", "Calendar year, e.g. 2024", 2024)],
        usage_guidance="OEE by plant for a year; quality_metrics_daily joined to plants."),
    example(
        "Which production lines have a defect rate above 5%?",
        f"SELECT pl.line_name, ROUND(100.0 * SUM(CASE WHEN pe.event_type = 'defect_detected' THEN 1 ELSE 0 END) "
        f"/ NULLIF(SUM(CASE WHEN pe.event_type = 'unit_produced' THEN 1 ELSE 0 END), 0), 2) AS defect_rate_pct "
        f"FROM {fqn}.production_events pe JOIN {fqn}.production_lines pl ON pe.production_line_id = pl.line_id "
        f"GROUP BY pl.line_name HAVING defect_rate_pct > 5 ORDER BY defect_rate_pct DESC",
        usage_guidance="Ratio with HAVING; production_events joined to production_lines."),
    example(
        "What is the scrap rate by state for {year}?",
        f"SELECT p.state, ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) AS scrap_rate_pct "
        f"FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id "
        f"WHERE YEAR(CAST(q.date AS DATE)) = :year GROUP BY p.state ORDER BY scrap_rate_pct DESC",
        parameters=[param("year", "NUMBER", "Calendar year", 2024)],
        usage_guidance="Scrap rate from quality_metrics_daily for date ranges."),
    example(
        "List all critical safety incidents",
        f"SELECT incident_id, incident_date, production_line_id, description, root_cause, corrective_action "
        f"FROM {fqn}.safety_incidents WHERE severity = 'Critical' ORDER BY incident_date DESC",
        usage_guidance="severity is Low/Medium/High/Critical."),
]
print(f"{len(examples)} example SQLs")

## 5. Text instructions — kept short on purpose

With the logic now in measures, joins, and examples, text instructions shrink to what only prose can express: **when to ask for clarification** and **formatting rules**. This is the opposite of the monolithic blob — a handful of lines, no SQL.

In [ ]:
sample_questions = sorted(
    [{"id": new_id(), "question": [q]} for q in [
        "What is the average OEE by plant for 2024?",
        "Which production lines have a defect rate above 5%?",
        "What is the scrap rate by state for 2024?",
    ]],
    key=lambda x: x["id"],
)

text_instructions = [{
    "id": new_id(),
    "content": [
        "When a question about a rate or metric omits a time range, ask which period (year or date range) before answering.\n",
        "Format percentages to 2 decimals; cast integer counts with CAST(... AS BIGINT).\n",
        "production_lines.status is static (Active/Maintenance) — never apply a date filter to it.\n",
        "For scrap rate over a date range, use quality_metrics_daily (scrap_count / units_produced), not production_events.\n",
    ],
}]
print(f"{len(text_instructions[0]['content'])} instruction lines (deliberately minimal)")

## 6. Assemble, validate, and create the agent

We sort every array (Genie requires it), check the Workbench constraints locally, then create the **primary** Knowledge Store agent and save it as `genie_space_id`.

In [ ]:
def _by_id(items):
    return sorted(items, key=lambda e: e["id"])


serialized = {
    "version": 2,
    "config": {"sample_questions": sample_questions},
    "data_sources": {"tables": tables},
    "instructions": {
        "text_instructions": text_instructions,
        "example_question_sqls": _by_id(examples),
        "join_specs": _by_id(joins),
        "sql_snippets": {
            "measures": _by_id(measures),
            "filters": _by_id(filters),
            "expressions": _by_id(expressions),
        },
    },
}


def validate(s):
    problems = []
    hexre = re.compile(r"^[0-9a-f]{32}$")

    def check(items, label):
        for it in items:
            if not hexre.match(it.get("id", "")):
                problems.append(f"{label}: bad id {it.get('id')!r}")
        ids = [it["id"] for it in items]
        if ids != sorted(ids):
            problems.append(f"{label}: array not sorted by id")

    ins = s["instructions"]
    check(ins["example_question_sqls"], "example_question_sqls")
    check(ins["join_specs"], "join_specs")
    for k in ("measures", "filters", "expressions"):
        check(ins["sql_snippets"][k], f"sql_snippets.{k}")
    check(s["config"]["sample_questions"], "sample_questions")

    if sum(len(ins["sql_snippets"][k]) for k in ("measures", "filters", "expressions")) > 100:
        problems.append(">100 sql_snippets")
    if len(ins["text_instructions"]) > 1:
        problems.append(">1 text_instructions")
    if len(s["data_sources"]["tables"]) > 30:
        problems.append(">30 tables")
    for j in ins["join_specs"]:
        if len(j["sql"]) != 2 or "--rt=FROM_RELATIONSHIP_TYPE_" not in j["sql"][1]:
            problems.append("join_spec missing --rt marker")
    idents = [t["identifier"] for t in s["data_sources"]["tables"]]
    if idents != sorted(idents):
        problems.append("tables not sorted by identifier")
    return problems


problems = validate(serialized)
print("Local validation:", "OK" if not problems else problems)
serialized_str = json.dumps(serialized)
print(f"serialized_space: {len(serialized_str):,} chars")

In [ ]:
def _list_spaces():
    r = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers)
    r.raise_for_status()
    return r.json().get("spaces", [])


def create_or_update_genie_space(title, description, serialized_space_str):
    for s in _list_spaces():
        if s.get("title") == title:
            sid = s.get("space_id") or s.get("id")
            pr = requests.patch(
                f"{host}/api/2.0/genie/spaces/{sid}", headers=headers,
                json={"title": title, "description": description,
                      "warehouse_id": warehouse_id, "serialized_space": serialized_space_str},
            )
            print(f"Updated existing: {title!r} -> {sid} ({pr.status_code})")
            if pr.status_code not in (200, 201):
                print("  ", pr.text[:400])
            return sid, genie_ui_room_url(sid)
    resp = requests.post(
        f"{host}/api/2.0/genie/spaces", headers=headers,
        json={"title": title, "description": description,
              "warehouse_id": warehouse_id, "serialized_space": serialized_space_str},
    )
    if resp.status_code not in (200, 201):
        raise RuntimeError(f"Genie create failed {resp.status_code}: {resp.text[:800]}")
    sid = resp.json().get("space_id") or resp.json().get("id")
    print(f"Created: {title!r} -> {sid}")
    return sid, genie_ui_room_url(sid)


cur_id, cur_url = create_or_update_genie_space(
    GENIE_TITLE_CURATED, GENIE_DESC_CURATED, serialized_str
)

save_config_keys([
    {"key": CFG_KEY_CURATED, "value": cur_id,
     "space_name": GENIE_TITLE_CURATED, "space_url": cur_url},
])

print()
print("=" * 70)
print("KNOWLEDGE STORE AGENT (primary) CREATED")
print("=" * 70)
print(f"  {cur_url}")
print("=" * 70)
print("Ask it the same question you asked the baseline in 03 —")
print("the curated joins, measures, and examples make the difference.")

## Next

- **05 — Benchmarks:** measure this agent's accuracy against ground-truth SQL.
- **08 — Compare:** Baseline vs. Metric View vs. **Knowledge Store**, head-to-head.

**Takeaway:** curation isn't one big instruction — it's a handful of small, structured, testable pieces. That's what turns a demo into an agent people trust.